# Filtraggio Classi (Allineamento Dataset)

L'ASL Citizen dataset non è perfettamente stratificato dalla fonte: il Test Set e il Validation Set contengono vocaboli (classi) che **non esistono** nel Training Set.

### Il Problema:
Se alleniamo la rete neurale a riconoscere `N` segni e la valutiamo su un dataset contenente `N+1` segni, il modello fallirà sempre per quell'unico segno sconosciuto. Dobbiamo ripulire il Test e Val Set.

### Soluzione:
Questo script esplora tutte le directory in `Dataset_Keypoints_Train`, crea un set di classi "conosciute", e poi elimina fisicamente i file `.npy` dalle directory Test e Val se appartengono a classi sconosciute.

In [ ]:
"""
Script di filtro classi per Test e Validation set.

Rimuove le cartelle (classi) da Dataset_Keypoints_Test e Dataset_Keypoints_Val
che NON sono presenti in Dataset_Keypoints_Train, in modo che il modello venga
valutato solo su classi su cui è stato effettivamente addestrato.

Uso:
    python -m src.data_preparation.filter_classes
"""

import os
import shutil

TRAIN_DIR = 'data/processed/Dataset_Keypoints_Train'
TEST_DIR = 'data/processed/Dataset_Keypoints_Test'
VAL_DIR = 'data/processed/Dataset_Keypoints_Val'


def get_classes(directory):
    """Restituisce il set di classi (nomi cartelle) presenti in una directory."""
    if not os.path.exists(directory):
        return set()
    return set(d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d)))


def filter_dataset(target_dir, train_classes, dataset_name):
    """Rimuove le classi non presenti nel training set."""
    target_classes = get_classes(target_dir)
    extra_classes = target_classes - train_classes

    if not extra_classes:
        print(f"  ✓ {dataset_name}: nessuna classe extra da rimuovere.")
        return 0

    print(f"\n  {dataset_name}: {len(extra_classes)} classi da rimuovere")

    # Conta i file che verranno rimossi
    total_files = 0
    for cls in sorted(extra_classes):
        cls_path = os.path.join(target_dir, cls)
        n_files = len([f for f in os.listdir(cls_path) if f.endswith('.npy')])
        total_files += n_files

    print(f"  File .npy che verranno rimossi: {total_files}")
    print(f"  Classi rimosse:")

    for cls in sorted(extra_classes):
        cls_path = os.path.join(target_dir, cls)
        n_files = len([f for f in os.listdir(cls_path) if f.endswith('.npy')])
        shutil.rmtree(cls_path)
        print(f"    ✗ {cls} ({n_files} file)")

    remaining = len(target_classes) - len(extra_classes)
    print(f"\n  Classi rimaste in {dataset_name}: {remaining}")
    return len(extra_classes)


def main():
    print("=" * 70)
    print("  FILTRO CLASSI — ALLINEAMENTO TEST/VAL AL TRAIN SET")
    print("=" * 70)

    train_classes = get_classes(TRAIN_DIR)
    print(f"\n  Classi nel Train Set: {len(train_classes)}")

    test_classes = get_classes(TEST_DIR)
    val_classes = get_classes(VAL_DIR)
    print(f"  Classi nel Test Set:  {len(test_classes)}")
    print(f"  Classi nel Val Set:   {len(val_classes)}")

    extra_test = test_classes - train_classes
    extra_val = val_classes - train_classes
    print(f"\n  Classi extra nel Test (non in Train): {len(extra_test)}")
    print(f"  Classi extra nel Val (non in Train):  {len(extra_val)}")

    if not extra_test and not extra_val:
        print("\n  ✓ Nessuna pulizia necessaria. Tutti i set sono già allineati.")
        return

    print(f"\n{'─' * 70}")

    removed_test = filter_dataset(TEST_DIR, train_classes, "Test Set")
    removed_val = filter_dataset(VAL_DIR, train_classes, "Val Set")

    # Verifica finale
    print(f"\n{'=' * 70}")
    print("  VERIFICA FINALE")
    print(f"{'=' * 70}")

    final_train = len(get_classes(TRAIN_DIR))
    final_test = len(get_classes(TEST_DIR))
    final_val = len(get_classes(VAL_DIR))

    print(f"  Train: {final_train} classi")
    print(f"  Test:  {final_test} classi")
    print(f"  Val:   {final_val} classi")

    # Controlla anche l'inverso: classi nel train ma non nel test/val
    missing_in_test = train_classes - get_classes(TEST_DIR)
    missing_in_val = train_classes - get_classes(VAL_DIR)

    if missing_in_test:
        print(f"\n  ⚠ {len(missing_in_test)} classi del Train non presenti nel Test")
    if missing_in_val:
        print(f"\n  ⚠ {len(missing_in_val)} classi del Train non presenti nel Val")

    print(f"\n  Rimossi: {removed_test} classi dal Test, {removed_val} classi dal Val")
    print("=" * 70)


if __name__ == "__main__":
    main()
